# DeiT-Tiny Single Image Inference v1

CPU baseline with detailed prints per block.


In [ ]:
print('DeiT Inference Notebook v1')
import os
import time
import numpy as np
import torch
import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform


In [ ]:
IMAGE_PATH = 'ps/deit/Puppy-Cover.jpg'
WEIGHTS_PATH = 'ps/deit/deit_tiny_patch16_224_10.pth'
DEVICE = 'cpu'
EXPORT_DIR = 'ps/deit/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)
print('[INFO] image =', IMAGE_PATH)
print('[INFO] weights =', WEIGHTS_PATH)
print('[INFO] device =', DEVICE)


In [ ]:
print('[STEP] Load model')
model = timm.create_model('deit_tiny_patch16_224', pretrained=False)
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu')
if isinstance(ckpt, dict):
    if 'model' in ckpt:
        state = ckpt['model']
    elif 'state_dict' in ckpt:
        state = ckpt['state_dict']
    else:
        state = ckpt
else:
    state = ckpt
model.load_state_dict(state, strict=False)
model.eval()
model.to(DEVICE)
print('[OK] Model loaded')


In [ ]:
print('[STEP] Preprocess image')
img = Image.open(IMAGE_PATH).convert('RGB')
cfg = resolve_data_config({}, model=model)
transform = create_transform(**cfg)
x = transform(img).unsqueeze(0).to(DEVICE)
print('[OK] Input tensor shape =', tuple(x.shape))


In [ ]:
with torch.no_grad():
    print('[STEP] Patch embedding')
    x = model.patch_embed(x)
    print('  patch_embed out =', tuple(x.shape))

    print('[STEP] Add cls token + pos embed')
    cls_token = model.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = x + model.pos_embed
    x = model.pos_drop(x)
    print('  tokens =', tuple(x.shape))

    for i, blk in enumerate(model.blocks):
        t0 = time.time()
        print(f'[BLOCK {i:02d}] start')
        x = blk(x)
        print(f'[BLOCK {i:02d}] done, shape={tuple(x.shape)}, dt={time.time()-t0:.3f}s')

    print('[STEP] Final norm + head')
    x = model.norm(x)
    logits = model.head(x[:, 0])
    print('  logits =', tuple(logits.shape))

    probs = torch.softmax(logits, dim=-1)
    top5 = torch.topk(probs, k=5, dim=-1)
    print('[RESULT] Top-5 indices:', top5.indices.cpu().numpy().tolist())
    print('[RESULT] Top-5 probs:', top5.values.cpu().numpy().tolist())


In [ ]:
# Optional: dump tokens after norm
DUMP_TOKENS = True
if DUMP_TOKENS:
    out_path = os.path.join(EXPORT_DIR, 'tokens_after_norm.npy')
    np.save(out_path, x.cpu().numpy())
    print('[OK] Saved', out_path)
